# AeroTrack — Part 10: Manual Identity Review (Lower-Threshold Experiment)

This copy imports the lower-threshold Part 9 registry into a separate SQLite database. It does not reuse the earlier baseline database.

Review candidate crop pairs carefully. Only choose SAME when the crops show the same physical person. Choose DIFFERENT when they are distinct people, and UNCERTAIN when evidence is insufficient.


## Section 1 — Paths and review settings

Run Parts 1, 7, 8 and 9 first. Part 10 reads the shared active-video configuration created by Part 1 and the Part 9 consolidated outputs.

In [ ]:
from pathlib import Path
from contextlib import contextmanager
from io import BytesIO
from datetime import datetime, timezone
import sqlite3
import json
import re
import time
import hashlib

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image as PILImage
from IPython.display import display, HTML, Video

PROJECT_DIR = Path.cwd().resolve()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

ACTIVE_VIDEO_CONFIG_PATH = OUTPUT_DIR / "active_video_config.json"
if not ACTIVE_VIDEO_CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Missing {ACTIVE_VIDEO_CONFIG_PATH}. Run Part 1 before Part 10."
    )
with open(ACTIVE_VIDEO_CONFIG_PATH, "r", encoding="utf-8") as file:
    ACTIVE_VIDEO_CONFIG = json.load(file)

VIDEO_ID = str(ACTIVE_VIDEO_CONFIG["video_id"])
VIDEO_OUTPUT_DIR = OUTPUT_DIR / VIDEO_ID
VIDEO_PATH = PROJECT_DIR / ACTIVE_VIDEO_CONFIG["video_path"]
TRACK_LOG_PATH = PROJECT_DIR / ACTIVE_VIDEO_CONFIG["track_log_path"]
SEGMENT_START_SECONDS = float(ACTIVE_VIDEO_CONFIG.get("segment_start_seconds", 0.0))
VIDEO_SOURCES = [{
    "video_id": VIDEO_ID,
    "video_path": VIDEO_PATH,
    "track_log_path": TRACK_LOG_PATH,
    "segment_start_seconds": SEGMENT_START_SECONDS
}]

PART7_DIR = VIDEO_OUTPUT_DIR / "part7_global_identity"
PART9_DIR = VIDEO_OUTPUT_DIR / "part9_tracklet_consolidation"
PART10_DIR = VIDEO_OUTPUT_DIR / "part10_identity_review"
PART10_DIR.mkdir(parents=True, exist_ok=True)

PART9_REGISTRY_PATH = PART9_DIR / "consolidated_identity_registry.csv"
CANDIDATES_PATH = PART9_DIR / "candidate_graph_with_new_global_ids.csv"
TRACKLET_QUALITY_PATH = PART9_DIR / "tracklet_quality_report.csv"
CROP_MANIFEST_PATH = PART7_DIR / "crop_manifest.csv"
CROP_DIR = PART7_DIR / "tracklet_crops"

# Use a separate database for the lower-threshold experiment. This prevents
# the earlier baseline Global ID assignments from masking the new Part 9 groups.
# Keep this database for subsequent manual reviews of this same experiment.
DB_PATH = PART10_DIR / f"{VIDEO_ID}_identity_registry_fixed_v3.sqlite"

REVIEW_SIMILARITY_MIN = 0.70
MAX_CROP_IMAGES_PER_TRACKLET = 3
VISIBILITY_GRACE_SECONDS = 1.0
CREATE_ANNOTATED_VIDEO = False

print("Active video:", VIDEO_ID)
print("Video:", VIDEO_PATH, "| Exists:", VIDEO_PATH.exists())
print("Part 9 registry:", PART9_REGISTRY_PATH.exists())
print("Candidate graph:", CANDIDATES_PATH.exists())
print("Tracklet quality report:", TRACKLET_QUALITY_PATH.exists())
print("Tracking log:", TRACK_LOG_PATH.exists())
print("Part 7 crop manifest:", CROP_MANIFEST_PATH.exists())
print("Persistent database:", DB_PATH)


## Section 2 — Load the Part 9 data and validate tracklets

We need the Part 9 consolidated registry, candidate graph, tracklet quality report and original tracking log.

The candidate graph supplies similarity-ranked pairs. The tracking log supplies frame-level detections for recalculating visible and cumulative counts.

In [ ]:
required_paths = [
    PART9_REGISTRY_PATH,
    CANDIDATES_PATH,
    TRACKLET_QUALITY_PATH,
    TRACK_LOG_PATH
]

missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Missing Part 9 inputs:\n" +
        "\n".join(str(path) for path in missing_paths)
    )

part9_registry = pd.read_csv(PART9_REGISTRY_PATH)
candidate_df = pd.read_csv(CANDIDATES_PATH)
track_quality = pd.read_csv(TRACKLET_QUALITY_PATH)
track_log = pd.read_csv(TRACK_LOG_PATH)

required_registry = {"global_id", "tracklet_key", "video_id", "track_id"}
required_candidates = {
    "tracklet_a", "tracklet_b", "multi_mean_similarity",
    "multi_max_similarity", "rank_for_a", "rank_for_b",
    "margin_for_a", "margin_for_b"
}
required_quality = {
    "tracklet_key", "video_id", "track_id",
    "first_frame", "last_frame", "first_seen_seconds",
    "last_seen_seconds", "observations", "crop_count"
}
required_log = {
    "frame", "track_id", "confidence", "x1", "y1", "x2", "y2"
}

for name, frame, required in [
    ("Part 9 registry", part9_registry, required_registry),
    ("Candidate graph", candidate_df, required_candidates),
    ("Tracklet quality", track_quality, required_quality),
    ("Tracking log", track_log, required_log)
]:
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")

for column in [
    "multi_mean_similarity", "multi_max_similarity",
    "rank_for_a", "rank_for_b", "margin_for_a", "margin_for_b"
]:
    candidate_df[column] = pd.to_numeric(
        candidate_df[column], errors="coerce"
    )

track_quality["tracklet_key"] = track_quality["tracklet_key"].astype(str)
track_quality["track_id"] = track_quality["track_id"].astype(int)
track_quality["first_frame"] = track_quality["first_frame"].astype(int)
track_quality["last_frame"] = track_quality["last_frame"].astype(int)
track_quality["observations"] = track_quality["observations"].astype(int)
track_quality["crop_count"] = pd.to_numeric(
    track_quality["crop_count"], errors="coerce"
).fillna(0).astype(int)

part9_registry["tracklet_key"] = part9_registry["tracklet_key"].astype(str)
part9_registry["global_id"] = part9_registry["global_id"].astype(str)

if "video_id" not in track_log.columns:
    if len(VIDEO_SOURCES) != 1:
        raise ValueError(
            "The tracking log has no video_id column. Add video_id to the "
            "log before processing multiple videos."
        )
    track_log["video_id"] = str(VIDEO_SOURCES[0]["video_id"])

track_log["video_id"] = track_log["video_id"].astype(str)
track_log["frame"] = pd.to_numeric(
    track_log["frame"], errors="coerce"
).astype(int)
track_log["track_id"] = pd.to_numeric(
    track_log["track_id"], errors="coerce"
).astype(int)

track_log["tracklet_key"] = (
    track_log["video_id"] + "::track_" + track_log["track_id"].astype(str)
)

source_by_video = {
    str(item["video_id"]): item
    for item in VIDEO_SOURCES
}

if len(source_by_video) != len(VIDEO_SOURCES):
    raise ValueError("Each configured video must have a unique video_id.")

unknown_videos = set(track_log["video_id"].unique()) - set(source_by_video)
if unknown_videos:
    raise ValueError(
        f"Tracking log contains videos without VIDEO_SOURCES entries: {unknown_videos}"
    )

track_meta = (
    track_quality.set_index("tracklet_key")
    .to_dict("index")
)

current_tracklet_keys = sorted(
    set(track_log["tracklet_key"].astype(str))
)

print("Videos:", track_log["video_id"].nunique())
print("Local tracklets:", len(current_tracklet_keys))
print("Part 9 Global IDs:", part9_registry["global_id"].nunique())
print("Candidate pairs:", len(candidate_df))
print("Tracklets with crops:", int((track_quality["crop_count"] > 0).sum()))

display(
    track_quality["evidence_status"]
    .value_counts()
    .rename_axis("evidence_status")
    .reset_index(name="tracklets")
)

## Section 3 — Load crop paths for visual review

The primary image source is the Part 7 `crop_manifest.csv`. It contains representative crops for each tracklet.

The interface shows up to three representative crops for each side of a candidate pair. If an image cannot be found, the interface will say so rather than showing an unrelated image.

In [ ]:
crop_manifest = pd.DataFrame()

if CROP_MANIFEST_PATH.exists():
    crop_manifest = pd.read_csv(CROP_MANIFEST_PATH)

    required_crop_columns = {"tracklet_key", "crop_path"}

    if not required_crop_columns.issubset(crop_manifest.columns):
        print("Crop manifest is missing expected columns.")
        crop_manifest = pd.DataFrame()
else:
    print("Part 7 crop manifest not found. Using crop-folder fallback.")


def get_crop_paths(tracklet_key, maximum=MAX_CROP_IMAGES_PER_TRACKLET):
    """Return existing representative crop paths for one tracklet."""
    paths = []

    if not crop_manifest.empty:
        rows = crop_manifest[
            crop_manifest["tracklet_key"].astype(str) == str(tracklet_key)
        ].copy()

        if "quality_score" in rows.columns:
            rows["quality_score"] = pd.to_numeric(
                rows["quality_score"], errors="coerce"
            ).fillna(0)
            rows = rows.sort_values("quality_score", ascending=False)

        if "frame" in rows.columns:
            rows = rows.drop_duplicates("frame")

        for value in rows["crop_path"].tolist():
            path = Path(str(value))
            if path.exists() and path.is_file():
                paths.append(path)

    # Fallback to the Part 7 crop folder structure.
    if not paths:
        try:
            video_id, local_part = str(tracklet_key).split("::", 1)
            track_id = int(local_part.replace("track_", ""))
            folder = CROP_DIR / video_id / f"track_{track_id:04d}"

            if folder.exists():
                paths = sorted(folder.glob("*.jpg"))
        except Exception:
            paths = []

    return paths[:maximum]


crop_counts = {
    key: len(get_crop_paths(key))
    for key in current_tracklet_keys
}

print("Tracklets with visible crop files:", sum(v > 0 for v in crop_counts.values()))
print("Tracklets without crop files:", sum(v == 0 for v in crop_counts.values()))

## Section 4 — Create the persistent SQLite identity registry

The first execution imports the Part 9 consolidated Global IDs. Subsequent executions do not overwrite existing assignments.

New tracklets receive a new Global ID. Merged IDs are retained as aliases, not deleted.

In [ ]:
@contextmanager
def database_connection():
    connection = sqlite3.connect(DB_PATH, timeout=30)
    connection.row_factory = sqlite3.Row
    connection.execute("PRAGMA foreign_keys = ON")

    try:
        yield connection
        connection.commit()
    except Exception:
        connection.rollback()
        raise
    finally:
        connection.close()


with database_connection() as con:
    con.executescript("""
    CREATE TABLE IF NOT EXISTS metadata (
        key TEXT PRIMARY KEY,
        value TEXT NOT NULL
    );

    CREATE TABLE IF NOT EXISTS identities (
        global_id TEXT PRIMARY KEY,
        created_order INTEGER NOT NULL UNIQUE,
        created_at TEXT NOT NULL,
        source TEXT NOT NULL,
        is_active INTEGER NOT NULL DEFAULT 1
    );

    CREATE TABLE IF NOT EXISTS tracklet_assignments (
        tracklet_key TEXT PRIMARY KEY,
        global_id TEXT NOT NULL,
        video_id TEXT NOT NULL,
        track_id INTEGER NOT NULL,
        first_frame INTEGER,
        last_frame INTEGER,
        first_seen_seconds REAL,
        last_seen_seconds REAL,
        updated_at TEXT NOT NULL
    );

    CREATE TABLE IF NOT EXISTS global_id_aliases (
        alias_global_id TEXT PRIMARY KEY,
        canonical_global_id TEXT NOT NULL,
        reason TEXT NOT NULL,
        created_at TEXT NOT NULL
    );

    CREATE TABLE IF NOT EXISTS manual_pair_decisions (
        pair_key TEXT PRIMARY KEY,
        tracklet_a TEXT NOT NULL,
        tracklet_b TEXT NOT NULL,
        decision TEXT NOT NULL,
        notes TEXT,
        updated_at TEXT NOT NULL
    );

    CREATE TABLE IF NOT EXISTS decision_history (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        action_type TEXT NOT NULL,
        tracklet_a TEXT,
        tracklet_b TEXT,
        before_decision_json TEXT,
        alias_global_id TEXT,
        split_tracklet_key TEXT,
        split_old_global_id TEXT,
        split_new_global_id TEXT,
        created_at TEXT NOT NULL,
        undone_at TEXT
    );
    """)

print("Persistent SQLite database ready:", DB_PATH)

## Section 5 — Seed the database once and register new tracklets

On the first run, the 54 Part 9 tracklets and their current provisional Global IDs are imported.

On later runs, existing assignments are preserved. New tracklets receive new IDs; current Part 9 numbering never overwrites existing persistent mappings.

In [ ]:
def natural_global_id_key(global_id):
    match = re.search(r"(\d+)$", str(global_id))
    return int(match.group(1)) if match else 10**12


def allocate_global_id(con, source="new_tracklet"):
    rows = con.execute(
        "SELECT global_id FROM identities"
    ).fetchall()

    maximum = max(
        [natural_global_id_key(row["global_id"]) for row in rows],
        default=0
    )

    new_id = f"G{maximum + 1:04d}"

    next_order = con.execute(
        "SELECT COALESCE(MAX(created_order), 0) + 1 AS next_order "
        "FROM identities"
    ).fetchone()["next_order"]

    con.execute(
        """
        INSERT INTO identities
        (global_id, created_order, created_at, source, is_active)
        VALUES (?, ?, ?, ?, 1)
        """,
        (
            new_id,
            int(next_order),
            datetime.now(timezone.utc).isoformat(),
            source
        )
    )

    return new_id


def upsert_tracklet_metadata(con, key, global_id, meta):
    con.execute(
        """
        INSERT INTO tracklet_assignments (
            tracklet_key, global_id, video_id, track_id,
            first_frame, last_frame, first_seen_seconds,
            last_seen_seconds, updated_at
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(tracklet_key) DO UPDATE SET
            video_id=excluded.video_id,
            track_id=excluded.track_id,
            first_frame=excluded.first_frame,
            last_frame=excluded.last_frame,
            first_seen_seconds=excluded.first_seen_seconds,
            last_seen_seconds=excluded.last_seen_seconds,
            updated_at=excluded.updated_at
        """,
        (
            key,
            global_id,
            str(meta["video_id"]),
            int(meta["track_id"]),
            int(meta["first_frame"]),
            int(meta["last_frame"]),
            float(meta["first_seen_seconds"]),
            float(meta["last_seen_seconds"]),
            datetime.now(timezone.utc).isoformat()
        )
    )


with database_connection() as con:
    initialized = con.execute(
        "SELECT value FROM metadata WHERE key='initialized_from_part9'"
    ).fetchone()

    if initialized is None:
        unique_ids = sorted(
            part9_registry["global_id"].astype(str).unique(),
            key=natural_global_id_key
        )

        for order, global_id in enumerate(unique_ids, start=1):
            con.execute(
                """
                INSERT OR IGNORE INTO identities
                (global_id, created_order, created_at, source, is_active)
                VALUES (?, ?, ?, 'part9_import', 1)
                """,
                (
                    global_id,
                    order,
                    datetime.now(timezone.utc).isoformat()
                )
            )

        for _, row in part9_registry.iterrows():
            key = str(row["tracklet_key"])
            global_id = str(row["global_id"])
            meta = track_meta.get(key)

            if meta is not None:
                upsert_tracklet_metadata(con, key, global_id, meta)

        con.execute(
            """
            INSERT OR REPLACE INTO metadata(key, value)
            VALUES ('initialized_from_part9', 'true')
            """
        )

        print("Initial persistent registry imported from Part 9.")
    else:
        print("Existing persistent registry found. Existing IDs will be preserved.")

    existing_keys = {
        row["tracklet_key"]
        for row in con.execute(
            "SELECT tracklet_key FROM tracklet_assignments"
        ).fetchall()
    }

    new_tracklets = [
        key for key in current_tracklet_keys
        if key not in existing_keys
    ]

    for key in new_tracklets:
        meta = track_meta.get(key)
        if meta is None:
            continue

        new_id = allocate_global_id(con, source="new_tracklet")
        upsert_tracklet_metadata(con, key, new_id, meta)

    # Update descriptive metadata, but never overwrite existing Global IDs.
    for key in current_tracklet_keys:
        meta = track_meta.get(key)
        if meta is None:
            continue

        row = con.execute(
            "SELECT global_id FROM tracklet_assignments WHERE tracklet_key=?",
            (key,)
        ).fetchone()

        if row is not None:
            upsert_tracklet_metadata(con, key, row["global_id"], meta)

    print("New tracklets assigned:", len(new_tracklets))

with database_connection() as con:
    print(
        "Persistent tracklet assignments:",
        con.execute("SELECT COUNT(*) AS n FROM tracklet_assignments").fetchone()["n"]
    )
    print(
        "Persistent Global IDs:",
        con.execute("SELECT COUNT(*) AS n FROM identities").fetchone()["n"]
    )

## Section 6 — Define Global ID, merge, split and undo operations

A manual merge chooses the older persistent Global ID as the canonical ID. The other ID becomes an alias.

Before merging, the notebook checks for same-video temporal overlap and previously rejected (`different`) pairs.

In [ ]:
def pair_key_for(tracklet_a, tracklet_b):
    a, b = sorted([str(tracklet_a), str(tracklet_b)])
    return a + "\x1f" + b


def get_pair_decision(con, tracklet_a, tracklet_b):
    key = pair_key_for(tracklet_a, tracklet_b)
    row = con.execute(
        "SELECT * FROM manual_pair_decisions WHERE pair_key=?",
        (key,)
    ).fetchone()
    return dict(row) if row is not None else None


def restore_pair_decision(con, pair_key, previous_json):
    if previous_json is None:
        con.execute(
            "DELETE FROM manual_pair_decisions WHERE pair_key=?",
            (pair_key,)
        )
        return

    old = json.loads(previous_json)

    con.execute(
        """
        INSERT OR REPLACE INTO manual_pair_decisions
        (pair_key, tracklet_a, tracklet_b, decision, notes, updated_at)
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        (
            pair_key,
            old["tracklet_a"],
            old["tracklet_b"],
            old["decision"],
            old.get("notes", ""),
            old.get("updated_at", datetime.now(timezone.utc).isoformat())
        )
    )


def resolve_global_id(con, global_id):
    current = str(global_id)
    visited = set()

    while True:
        if current in visited:
            raise RuntimeError("Circular Global ID alias detected.")
        visited.add(current)

        row = con.execute(
            "SELECT canonical_global_id FROM global_id_aliases "
            "WHERE alias_global_id=?",
            (current,)
        ).fetchone()

        if row is None:
            return current

        current = str(row["canonical_global_id"])


def current_assignment_map(con, only_current=True):
    rows = con.execute(
        "SELECT tracklet_key, global_id FROM tracklet_assignments"
    ).fetchall()

    result = {}
    current_keys = set(current_tracklet_keys)

    for row in rows:
        key = str(row["tracklet_key"])

        if only_current and key not in current_keys:
            continue

        result[key] = resolve_global_id(con, row["global_id"])

    return result


def identity_created_order(con, global_id):
    row = con.execute(
        "SELECT created_order FROM identities WHERE global_id=?",
        (global_id,)
    ).fetchone()
    return int(row["created_order"]) if row else 10**12


def members_for_global_id(con, global_id):
    target = resolve_global_id(con, global_id)
    rows = con.execute(
        "SELECT tracklet_key, global_id FROM tracklet_assignments"
    ).fetchall()

    return [
        str(row["tracklet_key"])
        for row in rows
        if resolve_global_id(con, row["global_id"]) == target
    ]


def check_merge_safety(con, global_a, global_b, ignored_pair_key=None):
    group_a = set(members_for_global_id(con, global_a))
    group_b = set(members_for_global_id(con, global_b))

    # A person cannot be represented by two simultaneous tracklets in one video.
    for key_a in group_a:
        meta_a = track_meta.get(key_a)
        if meta_a is None:
            continue

        for key_b in group_b:
            meta_b = track_meta.get(key_b)
            if meta_b is None:
                continue

            if str(meta_a["video_id"]) != str(meta_b["video_id"]):
                continue

            overlaps = not (
                int(meta_a["last_frame"]) < int(meta_b["first_frame"]) or
                int(meta_b["last_frame"]) < int(meta_a["first_frame"])
            )

            if overlaps:
                return False, (
                    "Merge blocked: these groups contain tracklets that "
                    f"overlap in {meta_a['video_id']}: {key_a} and {key_b}."
                )

    # Respect manually rejected pairs across both groups.
    different_rows = con.execute(
        "SELECT pair_key, tracklet_a, tracklet_b "
        "FROM manual_pair_decisions WHERE decision='different'"
    ).fetchall()

    for row in different_rows:
        if ignored_pair_key and row["pair_key"] == ignored_pair_key:
            continue

        a = str(row["tracklet_a"])
        b = str(row["tracklet_b"])

        conflict = (
            (a in group_a and b in group_b) or
            (a in group_b and b in group_a)
        )

        if conflict:
            return False, (
                "Merge blocked by a previous manual 'different' decision "
                f"between {a} and {b}."
            )

    return True, "Merge is compatible with the current safeguards."


def choose_canonical_id(con, global_a, global_b):
    order_a = identity_created_order(con, global_a)
    order_b = identity_created_order(con, global_b)

    if (order_a, global_a) <= (order_b, global_b):
        return global_a, global_b

    return global_b, global_a


def append_history(
    con,
    action_type,
    tracklet_a=None,
    tracklet_b=None,
    before_decision_json=None,
    alias_global_id=None,
    split_tracklet_key=None,
    split_old_global_id=None,
    split_new_global_id=None
):
    con.execute(
        """
        INSERT INTO decision_history (
            action_type, tracklet_a, tracklet_b,
            before_decision_json, alias_global_id,
            split_tracklet_key, split_old_global_id,
            split_new_global_id, created_at, undone_at
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, NULL)
        """,
        (
            action_type,
            tracklet_a,
            tracklet_b,
            before_decision_json,
            alias_global_id,
            split_tracklet_key,
            split_old_global_id,
            split_new_global_id,
            datetime.now(timezone.utc).isoformat()
        )
    )


def record_pair_decision(tracklet_a, tracklet_b, decision, notes=""):
    tracklet_a = str(tracklet_a)
    tracklet_b = str(tracklet_b)

    if tracklet_a == tracklet_b:
        raise ValueError("Choose two different tracklets.")

    if decision not in {"same", "different", "uncertain"}:
        raise ValueError("Decision must be same, different or uncertain.")

    pair_key = pair_key_for(tracklet_a, tracklet_b)
    alias_created = None

    with database_connection() as con:
        assignment_a = con.execute(
            "SELECT global_id FROM tracklet_assignments WHERE tracklet_key=?",
            (tracklet_a,)
        ).fetchone()
        assignment_b = con.execute(
            "SELECT global_id FROM tracklet_assignments WHERE tracklet_key=?",
            (tracklet_b,)
        ).fetchone()

        if assignment_a is None or assignment_b is None:
            raise KeyError("One selected tracklet is not in the persistent registry.")

        before = get_pair_decision(con, tracklet_a, tracklet_b)
        before_json = json.dumps(before) if before is not None else None

        gid_a = resolve_global_id(con, assignment_a["global_id"])
        gid_b = resolve_global_id(con, assignment_b["global_id"])

        if decision == "same" and gid_a != gid_b:
            safe, reason = check_merge_safety(
                con, gid_a, gid_b, ignored_pair_key=pair_key
            )

            if not safe:
                raise ValueError(reason)

            canonical, alias = choose_canonical_id(con, gid_a, gid_b)

            con.execute(
                """
                INSERT OR REPLACE INTO global_id_aliases
                (alias_global_id, canonical_global_id, reason, created_at)
                VALUES (?, ?, 'manual_same_decision', ?)
                """,
                (
                    alias,
                    canonical,
                    datetime.now(timezone.utc).isoformat()
                )
            )

            alias_created = alias

        con.execute(
            """
            INSERT OR REPLACE INTO manual_pair_decisions
            (pair_key, tracklet_a, tracklet_b, decision, notes, updated_at)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (
                pair_key,
                tracklet_a,
                tracklet_b,
                decision,
                notes,
                datetime.now(timezone.utc).isoformat()
            )
        )

        append_history(
            con,
            action_type="pair_decision",
            tracklet_a=tracklet_a,
            tracklet_b=tracklet_b,
            before_decision_json=before_json,
            alias_global_id=alias_created
        )

    export_persistent_state()

    with database_connection() as con:
        gid_a_now = resolve_global_id(
            con,
            con.execute(
                "SELECT global_id FROM tracklet_assignments WHERE tracklet_key=?",
                (tracklet_a,)
            ).fetchone()["global_id"]
        )
        gid_b_now = resolve_global_id(
            con,
            con.execute(
                "SELECT global_id FROM tracklet_assignments WHERE tracklet_key=?",
                (tracklet_b,)
            ).fetchone()["global_id"]
        )

    if decision == "same":
        return (
            f"Saved SAME decision. Current Global IDs: "
            f"{gid_a_now} and {gid_b_now}."
        )

    if decision == "different" and gid_a_now == gid_b_now:
        return (
            "Saved DIFFERENT decision, but these tracklets are already in "
            "the same Global ID group. Use 'Separate selected tracklet' to split one."
        )

    return f"Saved {decision.upper()} decision. No Global ID merge was performed."


def split_tracklet_from_group(tracklet_key):
    tracklet_key = str(tracklet_key)

    with database_connection() as con:
        row = con.execute(
            "SELECT global_id FROM tracklet_assignments WHERE tracklet_key=?",
            (tracklet_key,)
        ).fetchone()

        if row is None:
            raise KeyError(f"Tracklet not found: {tracklet_key}")

        old_raw_gid = str(row["global_id"])
        old_canonical_gid = resolve_global_id(con, old_raw_gid)
        members = members_for_global_id(con, old_canonical_gid)

        if len(members) <= 1:
            raise ValueError(
                f"{tracklet_key} is already the only tracklet in {old_canonical_gid}."
            )

        new_gid = allocate_global_id(con, source="manual_split")

        con.execute(
            "UPDATE tracklet_assignments SET global_id=?, updated_at=? "
            "WHERE tracklet_key=?",
            (
                new_gid,
                datetime.now(timezone.utc).isoformat(),
                tracklet_key
            )
        )

        append_history(
            con,
            action_type="split",
            split_tracklet_key=tracklet_key,
            split_old_global_id=old_raw_gid,
            split_new_global_id=new_gid
        )

    export_persistent_state()
    return f"Separated {tracklet_key} into new Global ID {new_gid}."


def undo_last_action():
    with database_connection() as con:
        row = con.execute(
            """
            SELECT * FROM decision_history
            WHERE undone_at IS NULL
            ORDER BY id DESC
            LIMIT 1
            """
        ).fetchone()

        if row is None:
            return "There are no actions available to undo."

        if row["action_type"] == "pair_decision":
            if row["alias_global_id"]:
                con.execute(
                    "DELETE FROM global_id_aliases WHERE alias_global_id=?",
                    (row["alias_global_id"],)
                )

            pair_key = pair_key_for(
                row["tracklet_a"], row["tracklet_b"]
            )
            restore_pair_decision(
                con,
                pair_key,
                row["before_decision_json"]
            )

        elif row["action_type"] == "split":
            con.execute(
                "UPDATE tracklet_assignments SET global_id=?, updated_at=? "
                "WHERE tracklet_key=?",
                (
                    row["split_old_global_id"],
                    datetime.now(timezone.utc).isoformat(),
                    row["split_tracklet_key"]
                )
            )

            con.execute(
                "UPDATE identities SET is_active=0 WHERE global_id=?",
                (row["split_new_global_id"],)
            )

        con.execute(
            "UPDATE decision_history SET undone_at=? WHERE id=?",
            (datetime.now(timezone.utc).isoformat(), int(row["id"]))
        )

        action_description = row["action_type"]

    export_persistent_state()
    return f"Undid the most recent {action_description} action."

## Section 7 — Export the persistent registry and decision history

This function saves review decisions and the current Global ID mapping after every button action.

The SQLite database remains the authoritative persistent store. CSV files are convenient for inspecting and sharing the current state.

In [ ]:
def export_persistent_state():
    with database_connection() as con:
        assignment_rows = con.execute(
            "SELECT * FROM tracklet_assignments"
        ).fetchall()

        assignment_records = []

        for row in assignment_rows:
            key = str(row["tracklet_key"])

            # Export current tracklets; retain older records in SQLite.
            if key not in set(current_tracklet_keys):
                continue

            gid = resolve_global_id(con, row["global_id"])
            quality = track_meta.get(key, {})

            assignment_records.append({
                "global_id": gid,
                "stored_global_id": str(row["global_id"]),
                "tracklet_key": key,
                "video_id": str(row["video_id"]),
                "track_id": int(row["track_id"]),
                "first_frame": row["first_frame"],
                "last_frame": row["last_frame"],
                "first_seen_seconds": row["first_seen_seconds"],
                "last_seen_seconds": row["last_seen_seconds"],
                "observations": quality.get("observations", np.nan),
                "crop_count": quality.get("crop_count", 0),
                "evidence_status": quality.get("evidence_status", "unknown")
            })

        alias_rows = con.execute(
            "SELECT * FROM global_id_aliases ORDER BY created_at"
        ).fetchall()

        decision_rows = con.execute(
            "SELECT * FROM manual_pair_decisions ORDER BY updated_at"
        ).fetchall()

        history_rows = con.execute(
            "SELECT * FROM decision_history ORDER BY id"
        ).fetchall()

    persistent_registry = pd.DataFrame(assignment_records)

    if persistent_registry.empty:
        raise RuntimeError("No current tracklet assignments found.")

    group_sizes = (
        persistent_registry.groupby("global_id")["tracklet_key"]
        .nunique()
        .to_dict()
    )

    manual_same_pairs = [
        row for row in decision_rows
        if row["decision"] == "same"
    ]

    manually_reviewed_ids = set()

    current_keys = set(persistent_registry["tracklet_key"])

    for row in manual_same_pairs:
        a = row["tracklet_a"]
        b = row["tracklet_b"]

        if a in current_keys and b in current_keys:
            gid_a = persistent_registry.loc[
                persistent_registry["tracklet_key"] == a, "global_id"
            ].iloc[0]
            gid_b = persistent_registry.loc[
                persistent_registry["tracklet_key"] == b, "global_id"
            ].iloc[0]

            if gid_a == gid_b:
                manually_reviewed_ids.add(gid_a)

    persistent_registry["group_tracklet_count"] = (
        persistent_registry["global_id"].map(group_sizes)
    )
    persistent_registry["has_human_approved_link"] = (
        persistent_registry["global_id"].isin(manually_reviewed_ids)
    )

    identity_groups = (
        persistent_registry.groupby("global_id")
        .agg(
            tracklet_count=("tracklet_key", "nunique"),
            video_count=("video_id", "nunique"),
            video_ids=("video_id", lambda values: ", ".join(sorted(set(values)))),
            tracklet_keys=("tracklet_key", lambda values: " | ".join(sorted(values))),
            tracker_ids=("track_id", lambda values: ", ".join(
                str(x) for x in sorted(set(values.astype(int)))
            )),
            first_seen_seconds=("first_seen_seconds", "min"),
            last_seen_seconds=("last_seen_seconds", "max"),
            total_observations=("observations", "sum"),
            has_human_approved_link=("has_human_approved_link", "max")
        )
        .reset_index()
    )

    # Keep column headers even when no aliases, decisions or history exist yet.
    # This prevents pandas EmptyDataError when a user has not reviewed any pairs.
    aliases_df = pd.DataFrame(
        [dict(row) for row in alias_rows],
        columns=[
            "alias_global_id", "canonical_global_id", "reason", "created_at"
        ]
    )
    decisions_df = pd.DataFrame(
        [dict(row) for row in decision_rows],
        columns=[
            "pair_key", "tracklet_a", "tracklet_b",
            "decision", "notes", "updated_at"
        ]
    )
    history_df = pd.DataFrame(
        [dict(row) for row in history_rows],
        columns=[
            "id", "action_type", "tracklet_a", "tracklet_b",
            "before_decision_json", "alias_global_id",
            "split_tracklet_key", "split_old_global_id",
            "split_new_global_id", "created_at", "undone_at"
        ]
    )

    persistent_registry.to_csv(
        PART10_DIR / "persistent_global_identity_registry.csv",
        index=False
    )
    identity_groups.to_csv(
        PART10_DIR / "persistent_identity_groups.csv",
        index=False
    )
    aliases_df.to_csv(
        PART10_DIR / "global_id_aliases.csv",
        index=False
    )
    decisions_df.to_csv(
        PART10_DIR / "manual_pair_decisions.csv",
        index=False
    )
    history_df.to_csv(
        PART10_DIR / "identity_review_audit_log.csv",
        index=False
    )

    return persistent_registry, identity_groups


persistent_registry, identity_groups = export_persistent_state()

print("Persistent registry exported.")
print("Tracklets:", len(persistent_registry))
print("Current Global IDs:", persistent_registry["global_id"].nunique())
print("Identity groups:", len(identity_groups))
print("SQLite database:", DB_PATH)

display(
    identity_groups.sort_values(
        ["tracklet_count", "total_observations"],
        ascending=False
    ).head(20)
)

## Section 8 — Prepare the manual review queue

The queue includes candidate pairs that:
- Have multi-mean similarity >= `REVIEW_SIMILARITY_MIN`.
- Currently belong to different persistent Global IDs.

Candidates already in the same Global ID group are excluded from the default queue. You can still compare any two tracklets using the custom selectors.

The default threshold is 0.70 so we see more than only the strongest candidates. Similarity is a ranking signal, not a probability.

In [ ]:
def get_current_gid_map():
    with database_connection() as con:
        return current_assignment_map(con, only_current=True)


current_gid_map = get_current_gid_map()

review_queue = candidate_df.copy()
review_queue["tracklet_a"] = review_queue["tracklet_a"].astype(str)
review_queue["tracklet_b"] = review_queue["tracklet_b"].astype(str)

review_queue["current_gid_a"] = review_queue["tracklet_a"].map(current_gid_map)
review_queue["current_gid_b"] = review_queue["tracklet_b"].map(current_gid_map)

review_queue = review_queue[
    (review_queue["multi_mean_similarity"] >= REVIEW_SIMILARITY_MIN) &
    (review_queue["current_gid_a"].notna()) &
    (review_queue["current_gid_b"].notna()) &
    (review_queue["current_gid_a"] != review_queue["current_gid_b"])
].copy()

review_queue = review_queue.sort_values(
    "multi_mean_similarity",
    ascending=False
).reset_index(drop=True)

print("Manual review pairs:", len(review_queue))
print("Similarity threshold:", REVIEW_SIMILARITY_MIN)

display(
    review_queue[
        [
            "tracklet_a", "tracklet_b",
            "current_gid_a", "current_gid_b",
            "multi_mean_similarity", "multi_max_similarity",
            "rank_for_a", "rank_for_b",
            "margin_for_a", "margin_for_b"
        ]
    ].head(30)
)

review_queue.to_csv(
    PART10_DIR / "manual_review_queue.csv",
    index=False
)

## Section 9 — Launch the interactive side-by-side review interface

Select a candidate pair, inspect up to three crops per tracklet, and make one of three decisions:

- **Same person:** merge their persistent Global ID groups.
- **Different people:** store a negative decision to prevent a future merge.
- **Uncertain:** save the pair for later review without merging.

The interface saves each decision immediately to SQLite and exports updated CSV files.

You can also navigate the candidate queue, compare any two tracklets, separate one tracklet from its current Global ID, and undo the most recent action.

In [ ]:
try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print(
        "ipywidgets is not installed. Run `%pip install ipywidgets`, "
        "restart the Jupyter kernel, and rerun this section. "
        "You can still edit the exported manual_pair_decisions.csv file."
    )


if WIDGETS_AVAILABLE:
    from IPython.display import clear_output

    def current_gid(tracklet_key):
        return get_current_gid_map().get(str(tracklet_key), "UNKNOWN")


    def make_tracklet_panel(tracklet_key):
        tracklet_key = str(tracklet_key)
        quality = track_meta.get(tracklet_key, {})
        gid = current_gid(tracklet_key)

        title = widgets.HTML(
            value=(
                f"<h3>{tracklet_key}</h3>"
                f"<b>Persistent Global ID:</b> {gid}<br>"
                f"<b>Frames:</b> {quality.get('first_frame', '?')}–"
                f"{quality.get('last_frame', '?')}<br>"
                f"<b>Observations:</b> {quality.get('observations', '?')} | "
                f"<b>ReID crops:</b> {quality.get('crop_count', 0)}"
            )
        )

        path_list = get_crop_paths(tracklet_key)
        image_items = []

        for index, path in enumerate(path_list, start=1):
            try:
                with PILImage.open(path) as source_image:
                    image = source_image.convert("RGB")
                    image.thumbnail((240, 240))

                    buffer = BytesIO()
                    image.save(buffer, format="JPEG", quality=90)
                    image_bytes = buffer.getvalue()

                image_widget = widgets.Image(
                    value=image_bytes,
                    format="jpeg",
                    layout=widgets.Layout(
                        width="180px",
                        height="190px"
                    )
                )

                image_items.append(
                    widgets.VBox([
                        widgets.HTML(
                            value=f"<small>Crop {index}: {path.name}</small>"
                        ),
                        image_widget
                    ])
                )
            except Exception as exc:
                image_items.append(
                    widgets.HTML(
                        value=f"<small>Could not load {path}: {exc}</small>"
                    )
                )

        if not image_items:
            image_items = [
                widgets.HTML(
                    value=(
                        "<i>No saved ReID crops found for this tracklet. "
                        "It may be a short tracklet without usable appearance data.</i>"
                    )
                )
            ]

        return widgets.VBox(
            [title, widgets.HBox(
                image_items,
                layout=widgets.Layout(flex_flow="row wrap")
            )],
            layout=widgets.Layout(
                width="48%",
                border="1px solid #999",
                padding="8px"
            )
        )


    queue_options = []

    for index, row in review_queue.iterrows():
        label = (
            f"{index + 1:02d} | {row['multi_mean_similarity']:.3f} | "
            f"{row['tracklet_a']} ({row['current_gid_a']}) ↔ "
            f"{row['tracklet_b']} ({row['current_gid_b']})"
        )
        queue_options.append((label, int(index)))

    pair_dropdown = widgets.Dropdown(
        options=queue_options,
        description="Candidate:",
        layout=widgets.Layout(width="98%"),
        style={"description_width": "85px"}
    )

    pair_notes = widgets.Textarea(
        value="",
        placeholder="Optional notes: clothing, location, visible evidence, etc.",
        description="Notes:",
        layout=widgets.Layout(width="98%", height="70px"),
        style={"description_width": "85px"}
    )

    comparison_output = widgets.Output(
        layout=widgets.Layout(width="100%")
    )
    action_status = widgets.HTML(value="<i>Select a candidate pair.</i>")

    previous_button = widgets.Button(description="Previous", icon="arrow-left")
    next_button = widgets.Button(description="Next", icon="arrow-right")
    same_button = widgets.Button(
        description="Same person — merge",
        button_style="success",
        icon="check"
    )
    different_button = widgets.Button(
        description="Different people",
        button_style="danger",
        icon="times"
    )
    uncertain_button = widgets.Button(
        description="Uncertain",
        button_style="warning",
        icon="question"
    )
    undo_button = widgets.Button(
        description="Undo last action",
        icon="undo"
    )

    active_pair = {"a": None, "b": None, "source": None}

    def set_active_pair(tracklet_a, tracklet_b, source="manual"):
        active_pair["a"] = str(tracklet_a)
        active_pair["b"] = str(tracklet_b)
        active_pair["source"] = source
        render_active_pair()


    def render_active_pair():
        key_a = active_pair["a"]
        key_b = active_pair["b"]

        with comparison_output:
            clear_output(wait=True)

            if not key_a or not key_b:
                display(HTML("<i>No pair selected.</i>"))
                return

            gid_a = current_gid(key_a)
            gid_b = current_gid(key_b)

            with database_connection() as con:
                decision = get_pair_decision(con, key_a, key_b)

            decision_text = (
                "Not reviewed"
                if decision is None
                else (
                    f"{decision['decision'].upper()} — "
                    f"{decision.get('notes', '')}"
                )
            )

            pair_info = widgets.HTML(
                value=(
                    f"<h3>Reviewing: {key_a} ↔ {key_b}</h3>"
                    f"<b>Current Global IDs:</b> {gid_a} and {gid_b}<br>"
                    f"<b>Previous decision:</b> {decision_text}<br>"
                    f"<b>Action:</b> {active_pair['source']}"
                )
            )

            panel_a = make_tracklet_panel(key_a)
            panel_b = make_tracklet_panel(key_b)

            display(pair_info)
            display(widgets.HBox([panel_a, panel_b]))


    def select_queue_pair(index):
        if review_queue.empty:
            active_pair["a"] = None
            active_pair["b"] = None
            render_active_pair()
            return

        index = max(0, min(int(index), len(review_queue) - 1))
        row = review_queue.iloc[index]

        set_active_pair(
            row["tracklet_a"],
            row["tracklet_b"],
            source=f"candidate {index + 1}/{len(review_queue)}"
        )


    def on_pair_dropdown_change(change):
        if change["name"] == "value" and change["new"] is not None:
            select_queue_pair(change["new"])


    pair_dropdown.observe(on_pair_dropdown_change, names="value")


    def move_queue(step):
        if review_queue.empty:
            return

        current_index = (
            int(pair_dropdown.value)
            if pair_dropdown.value is not None else 0
        )
        new_index = (current_index + step) % len(review_queue)
        pair_dropdown.value = new_index


    previous_button.on_click(lambda _: move_queue(-1))
    next_button.on_click(lambda _: move_queue(1))


    def handle_decision(decision):
        if not active_pair["a"] or not active_pair["b"]:
            action_status.value = "<b>Select a pair first.</b>"
            return

        try:
            message = record_pair_decision(
                active_pair["a"],
                active_pair["b"],
                decision,
                notes=pair_notes.value
            )
            action_status.value = (
                f"<span style='color:green'><b>{message}</b></span>"
            )
            render_active_pair()
        except Exception as exc:
            action_status.value = (
                f"<span style='color:red'><b>Not saved:</b> {exc}</span>"
            )


    same_button.on_click(lambda _: handle_decision("same"))
    different_button.on_click(lambda _: handle_decision("different"))
    uncertain_button.on_click(lambda _: handle_decision("uncertain"))


    def handle_undo(_):
        try:
            message = undo_last_action()
            action_status.value = (
                f"<span style='color:green'><b>{message}</b></span>"
            )
            render_active_pair()
        except Exception as exc:
            action_status.value = (
                f"<span style='color:red'><b>Undo failed:</b> {exc}</span>"
            )


    undo_button.on_click(handle_undo)

    # Custom selectors allow comparison of any tracklets, not only candidates.
    current_options = [
        (f"{key} [{current_gid_map.get(key, 'UNKNOWN')}]", key)
        for key in current_tracklet_keys
    ]

    custom_a = widgets.Dropdown(
        options=current_options,
        description="Tracklet A:",
        layout=widgets.Layout(width="48%"),
        style={"description_width": "85px"}
    )
    custom_b = widgets.Dropdown(
        options=current_options,
        description="Tracklet B:",
        layout=widgets.Layout(width="48%"),
        style={"description_width": "85px"}
    )

    if len(current_options) > 1:
        custom_b.value = current_options[1][1]

    custom_compare_button = widgets.Button(
        description="Compare custom pair",
        button_style="info",
        icon="search"
    )


    def on_custom_compare(_):
        if custom_a.value == custom_b.value:
            action_status.value = "<b>Choose two different tracklets.</b>"
            return

        pair_notes.value = ""
        set_active_pair(
            custom_a.value,
            custom_b.value,
            source="custom pair"
        )


    custom_compare_button.on_click(on_custom_compare)

    split_choice = widgets.Dropdown(
        options=[
            ("Tracklet A", "a"),
            ("Tracklet B", "b")
        ],
        description="Separate:",
        layout=widgets.Layout(width="220px"),
        style={"description_width": "70px"}
    )

    split_button = widgets.Button(
        description="Separate selected tracklet into a new Global ID",
        button_style="warning",
        icon="cut"
    )


    def on_split(_):
        side = split_choice.value
        key = active_pair.get(side)

        if not key:
            action_status.value = "<b>Select a pair first.</b>"
            return

        try:
            message = split_tracklet_from_group(key)
            action_status.value = (
                f"<span style='color:green'><b>{message}</b></span>"
            )
            render_active_pair()
        except Exception as exc:
            action_status.value = (
                f"<span style='color:red'><b>Split not performed:</b> {exc}</span>"
            )


    split_button.on_click(on_split)

    review_ui = widgets.VBox([
        widgets.HTML(
            value=(
                "<h2>AeroTrack Manual Identity Review</h2>"
                "<p>Inspect the two tracklets, then choose whether they show "
                "the same person, different people, or remain uncertain. "
                "Every decision is saved immediately.</p>"
            )
        ),
        pair_dropdown,
        comparison_output,
        pair_notes,
        widgets.HBox([
            previous_button,
            next_button,
            same_button,
            different_button,
            uncertain_button,
            undo_button
        ], layout=widgets.Layout(flex_flow="row wrap")),
        action_status,
        widgets.HTML("<hr><h3>Compare any two tracklets</h3>"),
        widgets.HBox([custom_a, custom_b]),
        custom_compare_button,
        widgets.HTML("<hr><h3>Undo an incorrect identity merge</h3>"),
        widgets.HTML(
            "<p>Select which side of the currently displayed pair should "
            "be detached from its current Global ID.</p>"
        ),
        widgets.HBox([split_choice, split_button])
    ])

    display(review_ui)

    if not review_queue.empty:
        select_queue_pair(0)
    else:
        with comparison_output:
            display(HTML(
                "<b>No unresolved candidate pairs at this threshold. "
                "Use the custom selectors to compare any two tracklets.</b>"
            ))

else:
    print(
        "Interactive controls are unavailable. Use the CSV workflow in "
        "Section 10 after installing ipywidgets."
    )

## Section 10 — CSV fallback and manual-decision import

The interactive interface saves decisions immediately. If `ipywidgets` is not available, you can edit:

`outputs/<video_id>/part10_identity_review/manual_pair_decisions.csv`

Use exactly `same`, `different` or `uncertain`, then run the import cell.

In [ ]:
def import_manual_decisions_from_csv(csv_path=None):
    csv_path = (
        Path(csv_path)
        if csv_path is not None
        else PART10_DIR / "manual_pair_decisions.csv"
    )

    if not csv_path.exists():
        raise FileNotFoundError(csv_path)

    decisions = pd.read_csv(csv_path)

    required = {"tracklet_a", "tracklet_b", "decision"}
    missing = required - set(decisions.columns)

    if missing:
        raise ValueError(f"Manual decision CSV missing columns: {missing}")

    applied = []
    skipped = []

    for _, row in decisions.iterrows():
        decision = str(row["decision"]).strip().lower()

        if decision not in {"same", "different", "uncertain"}:
            continue

        try:
            message = record_pair_decision(
                str(row["tracklet_a"]),
                str(row["tracklet_b"]),
                decision,
                notes=str(row.get("notes", ""))
            )
            applied.append({
                "tracklet_a": row["tracklet_a"],
                "tracklet_b": row["tracklet_b"],
                "decision": decision,
                "result": message
            })
        except Exception as exc:
            skipped.append({
                "tracklet_a": row["tracklet_a"],
                "tracklet_b": row["tracklet_b"],
                "decision": decision,
                "reason": str(exc)
            })

    print("Decisions applied:", len(applied))
    print("Decisions skipped:", len(skipped))

    if applied:
        display(pd.DataFrame(applied))

    if skipped:
        display(pd.DataFrame(skipped))


# Uncomment and run only after manually editing the CSV.
# import_manual_decisions_from_csv()

## Section 11 — Review the persistent identity groups

After making decisions, execute this section to see the updated persistent registry.

`has_human_approved_link` means at least one manually approved pair exists inside that group. It does not mean every member was individually verified.

In [ ]:
persistent_registry, identity_groups = export_persistent_state()

print("Current tracklet assignments:", len(persistent_registry))
print("Persistent Global IDs:", persistent_registry["global_id"].nunique())
print("Groups containing multiple tracklets:", int(
    (identity_groups["tracklet_count"] > 1).sum()
))
print("Groups with at least one human-approved link:", int(
    identity_groups["has_human_approved_link"].sum()
))

display(
    identity_groups.sort_values(
        ["tracklet_count", "total_observations"],
        ascending=False
    ).head(30)
)

print("\nTracklets that still need more ReID evidence:")
display(
    persistent_registry[
        persistent_registry["evidence_status"] != "reid_eligible"
    ][
        [
            "global_id", "tracklet_key", "observations",
            "crop_count", "evidence_status"
        ]
    ].sort_values("observations")
)

## Section 12 — Recalculate visible and cumulative unique-person counts

This section uses the persistent assignments after manual review.

For each frame in the active video:
- `visible_people`: distinct Global IDs detected in that exact frame.
- `recently_active_people`: distinct Global IDs seen within the previous grace period.
- `cumulative_unique_people`: distinct Global IDs observed at least once so far.

The count remains provisional because false detections, missed detections and unresolved tracklets still affect it.

In [ ]:
def calculate_count_timelines():
    assignment_map = get_current_gid_map()

    counting_log = track_log.copy()
    counting_log["global_id"] = counting_log["tracklet_key"].map(assignment_map)

    if counting_log["global_id"].isna().any():
        missing_keys = counting_log.loc[
            counting_log["global_id"].isna(), "tracklet_key"
        ].unique().tolist()
        raise ValueError(
            f"Some current tracklets have no persistent Global ID: {missing_keys[:10]}"
        )

    metadata = {}

    for source in VIDEO_SOURCES:
        video_id = str(source["video_id"])
        video_path = Path(source["video_path"])

        if not video_path.exists():
            raise FileNotFoundError(video_path)

        cap = cv2.VideoCapture(str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()

        if fps <= 0:
            raise ValueError(f"Invalid FPS for {video_id}")

        video_log = counting_log[
            counting_log["video_id"] == video_id
        ]

        if "time_seconds" in video_log.columns:
            offsets = (
                pd.to_numeric(video_log["time_seconds"], errors="coerce")
                - video_log["frame"] / fps
            ).dropna()
            inferred_start = float(offsets.median()) if not offsets.empty else 0.0
        else:
            inferred_start = 0.0

        configured_start = source.get("segment_start_seconds")
        segment_start = (
            float(configured_start)
            if configured_start is not None
            else inferred_start
        )

        metadata[video_id] = {
            "video_path": video_path,
            "fps": float(fps),
            "width": width,
            "height": height,
            "segment_start_seconds": segment_start
        }

    timeline_frames = []

    for video_id, video_log in counting_log.groupby("video_id"):
        meta = metadata[video_id]
        fps = meta["fps"]
        grace_frames = max(
            1,
            int(round(VISIBILITY_GRACE_SECONDS * fps))
        )

        min_frame = int(video_log["frame"].min())
        max_frame = int(video_log["frame"].max())

        ids_by_frame = (
            video_log.groupby("frame")["global_id"]
            .apply(lambda values: set(values.astype(str)))
            .to_dict()
        )

        time_by_frame = (
            video_log.groupby("frame")["time_seconds"].median().to_dict()
            if "time_seconds" in video_log.columns
            else {}
        )

        seen = set()
        last_seen = {}

        for frame_number in range(min_frame, max_frame + 1):
            current_ids = ids_by_frame.get(frame_number, set())

            for global_id in current_ids:
                seen.add(global_id)
                last_seen[global_id] = frame_number

            active_ids = {
                global_id
                for global_id, last_frame in last_seen.items()
                if frame_number - last_frame <= grace_frames
            }

            time_seconds = time_by_frame.get(
                frame_number,
                meta["segment_start_seconds"] + frame_number / fps
            )

            timeline_frames.append({
                "video_id": video_id,
                "frame": frame_number,
                "time_seconds": float(time_seconds),
                "visible_people": len(current_ids),
                "recently_active_people": len(active_ids),
                "cumulative_unique_people": len(seen)
            })

    timeline = pd.DataFrame(timeline_frames)

    summary_rows = []

    for video_id, group in timeline.groupby("video_id"):
        last = group.iloc[-1]
        video_log = counting_log[
            counting_log["video_id"] == video_id
        ]

        summary_rows.append({
            "video_id": video_id,
            "local_tracklets": int(video_log["tracklet_key"].nunique()),
            "provisional_unique_people": int(video_log["global_id"].nunique()),
            "visible_people_final_frame": int(last["visible_people"]),
            "recently_active_final_frame": int(
                last["recently_active_people"]
            ),
            "max_concurrent_visible_people": int(group["visible_people"].max()),
            "average_visible_people": float(group["visible_people"].mean()),
            "cumulative_unique_final": int(last["cumulative_unique_people"])
        })

    return counting_log, timeline, pd.DataFrame(summary_rows), metadata


counting_log, count_timeline, count_summary, video_metadata = (
    calculate_count_timelines()
)

TIMELINE_PATH = PART10_DIR / "manual_review_count_timeline.csv"
COUNT_SUMMARY_PATH = PART10_DIR / "manual_review_count_summary.csv"

count_timeline.to_csv(TIMELINE_PATH, index=False)
count_summary.to_csv(COUNT_SUMMARY_PATH, index=False)

print("UPDATED COUNT SUMMARY")
display(count_summary)
print("Total Global IDs across configured videos:",
      counting_log["global_id"].nunique())
print("Count timeline saved:", TIMELINE_PATH)

## Section 13 — Create the post-review dashboard

The dashboard compares currently visible people, recently active people and cumulative unique identities after manual decisions.

In [ ]:
DASHBOARD_PATHS = []

for video_id, timeline in count_timeline.groupby("video_id"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(
        timeline["time_seconds"],
        timeline["visible_people"],
        label="Visible in exact frame"
    )
    axes[0].plot(
        timeline["time_seconds"],
        timeline["recently_active_people"],
        label=f"Seen within {VISIBILITY_GRACE_SECONDS:.1f}s"
    )
    axes[0].set_title("People visible / recently active")
    axes[0].set_xlabel("Video time (seconds)")
    axes[0].set_ylabel("People")
    axes[0].legend()
    axes[0].grid(True, alpha=0.25)

    axes[1].plot(
        timeline["time_seconds"],
        timeline["cumulative_unique_people"]
    )
    axes[1].set_title("Cumulative provisional unique identities")
    axes[1].set_xlabel("Video time (seconds)")
    axes[1].set_ylabel("Global IDs")
    axes[1].grid(True, alpha=0.25)

    fig.suptitle(f"AeroTrack Part 10 — {video_id}")
    fig.tight_layout()

    path = PART10_DIR / f"{video_id}_manual_review_dashboard.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    DASHBOARD_PATHS.append(path)

    plt.show()
    print("Saved:", path)
    plt.close(fig)

## Section 14 — Optional: render a new annotated video using persistent Global IDs

Set `CREATE_ANNOTATED_VIDEO = True` in Section 1 if you want a new MP4 after manual review.

The overlay shows persistent Global IDs, local tracker IDs, visible people and cumulative counts. Detection and ReID are not rerun.

In [ ]:
ANNOTATED_PATHS = []

if CREATE_ANNOTATED_VIDEO:
    def stable_color(identifier):
        digest = hashlib.md5(
            str(identifier).encode("utf-8")
        ).digest()
        return (
            int(70 + digest[0] % 170),
            int(70 + digest[1] % 170),
            int(70 + digest[2] % 170)
        )

    for video_id, video_log in counting_log.groupby("video_id"):
        meta = video_metadata[video_id]

        timeline = count_timeline[
            count_timeline["video_id"] == video_id
        ].set_index("frame")

        rows_by_frame = {
            int(frame): group
            for frame, group in video_log.groupby("frame")
        }

        max_frame = int(video_log["frame"].max())

        cap = cv2.VideoCapture(str(meta["video_path"]))
        cap.set(
            cv2.CAP_PROP_POS_MSEC,
            meta["segment_start_seconds"] * 1000
        )

        if not cap.isOpened():
            raise RuntimeError(f"Could not open video {video_id}")

        output_path = PART10_DIR / f"{video_id}_manual_review_global_ids.mp4"

        writer = cv2.VideoWriter(
            str(output_path),
            cv2.VideoWriter_fourcc(*"mp4v"),
            meta["fps"],
            (meta["width"], meta["height"])
        )

        if not writer.isOpened():
            cap.release()
            raise RuntimeError(f"Could not create {output_path}")

        frame_number = 0

        while frame_number < max_frame:
            success, frame = cap.read()
            if not success:
                break

            frame_number += 1

            if frame_number in timeline.index:
                count_row = timeline.loc[frame_number]
                visible = int(count_row["visible_people"])
                active = int(count_row["recently_active_people"])
                cumulative = int(count_row["cumulative_unique_people"])
            else:
                visible = active = cumulative = 0

            rows = rows_by_frame.get(frame_number)

            if rows is not None:
                for _, row in rows.iterrows():
                    x1 = max(0, int(round(row["x1"])))
                    y1 = max(0, int(round(row["y1"])))
                    x2 = min(meta["width"] - 1, int(round(row["x2"])))
                    y2 = min(meta["height"] - 1, int(round(row["y2"])))

                    if x2 <= x1 or y2 <= y1:
                        continue

                    gid = str(row["global_id"])
                    tracker_id = int(row["track_id"])
                    color = stable_color(gid)

                    cv2.rectangle(
                        frame, (x1, y1), (x2, y2), color, 2
                    )

                    cv2.putText(
                        frame,
                        f"{gid} | T{tracker_id}",
                        (x1, max(22, y1 - 7)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.65,
                        color,
                        2,
                        cv2.LINE_AA
                    )

            panel = frame.copy()
            cv2.rectangle(
                panel,
                (15, 15),
                (int(meta["width"] * 0.29), 175),
                (0, 0, 0),
                -1
            )
            frame = cv2.addWeighted(panel, 0.55, frame, 0.45, 0)

            panel_text = [
                f"AeroTrack Manual Review | {video_id}",
                f"Visible now: {visible}",
                f"Recently active: {active}",
                f"Unique IDs seen: {cumulative}",
                "Persistent Global IDs"
            ]

            for idx, text in enumerate(panel_text):
                cv2.putText(
                    frame,
                    text,
                    (28, 45 + idx * 27),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.65,
                    (255, 255, 255),
                    2,
                    cv2.LINE_AA
                )

            writer.write(frame)

        cap.release()
        writer.release()
        ANNOTATED_PATHS.append(output_path)
        print("Saved annotated video:", output_path)

else:
    print(
        "Annotated video rendering is disabled. "
        "Set CREATE_ANNOTATED_VIDEO=True in Section 1 to enable it."
    )

## Section 15 — Save the final summary and outputs

Keep the SQLite database. It is the persistent source of truth for Global IDs and manual decisions.

In [ ]:
persistent_registry, identity_groups = export_persistent_state()

assignment_map = get_current_gid_map()
final_review_queue = candidate_df.copy()

final_review_queue["persistent_gid_a"] = (
    final_review_queue["tracklet_a"].astype(str).map(assignment_map)
)
final_review_queue["persistent_gid_b"] = (
    final_review_queue["tracklet_b"].astype(str).map(assignment_map)
)
final_review_queue["same_persistent_global_id"] = (
    final_review_queue["persistent_gid_a"] ==
    final_review_queue["persistent_gid_b"]
)

final_review_queue.to_csv(
    PART10_DIR / "all_candidate_pairs_with_persistent_ids.csv",
    index=False
)

# SQLite is the source of truth. A zero-row decision table is valid.
with database_connection() as con:
    manual_decision_count = int(
        con.execute(
            "SELECT COUNT(*) AS decision_count FROM manual_pair_decisions"
        ).fetchone()["decision_count"]
    )

summary = {
    "videos_processed": sorted(count_summary["video_id"].astype(str).tolist()),
    "current_tracklets": int(len(persistent_registry)),
    "persistent_global_ids": int(persistent_registry["global_id"].nunique()),
    "multi_tracklet_groups": int(
        (identity_groups["tracklet_count"] > 1).sum()
    ),
    "groups_with_human_approved_links": int(
        identity_groups["has_human_approved_link"].sum()
    ),
    "manual_pair_decision_records": int(manual_decision_count),
    "count_summary": count_summary.to_dict(orient="records"),
    "database_path": str(DB_PATH),
    "dashboard_paths": [str(path) for path in DASHBOARD_PATHS],
    "annotated_video_paths": [str(path) for path in ANNOTATED_PATHS],
    "note": (
        "Global IDs are persistent provisional identity groups. Human-approved "
        "links are based on manual visual review. Counts remain affected by "
        "detection misses, false detections and unresolved tracklet fragmentation."
    )
}

SUMMARY_PATH = PART10_DIR / "part10_review_summary.json"

with open(SUMMARY_PATH, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

print("PART 10 SUMMARY")
display(pd.DataFrame([{
    "current_tracklets": summary["current_tracklets"],
    "persistent_global_ids": summary["persistent_global_ids"],
    "multi_tracklet_groups": summary["multi_tracklet_groups"],
    "groups_with_human_approved_links": summary[
        "groups_with_human_approved_links"
    ],
    "manual_pair_decision_records": summary["manual_pair_decision_records"]
}]))

display(count_summary)
print("Persistent database:", DB_PATH)
print("Summary:", SUMMARY_PATH)

## Section 16 — What to send back

After reviewing a few candidate pairs, send:
1. The Part 10 summary.
2. `persistent_identity_groups.csv`.
3. `manual_pair_decisions.csv`.
4. A screenshot of the side-by-side review interface.
5. The updated count summary and dashboard.

For your first review session, focus on the highest-similarity unresolved candidates. You do not need to review every pair.

**Part 10 is the persistent identity-management layer.** It separates temporary tracker IDs from long-lived Global IDs and lets you correct identity assignments without retraining the models.